In [1]:
from delta import *
from pyspark.sql import *
from pyspark.sql.functions import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("create-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5c9f9885-2e8f-4959-8740-095b737719e3;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (888ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (45ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (71ms)
:: resolution report :: resolve 2052ms :: artifacts dl 1008ms
	

In [2]:
%%sparksql
CREATE OR REPLACE TABLE default.movie_and_show_titles (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/movie_and_show_titles';

25/05/28 15:14:32 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [3]:
deltaTable_titles = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/movie_and_show_titles")

deltaTable_titles.toDF().show(5)

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [4]:
df_netflix = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")
df_netflix_deduped = df_netflix.dropDuplicates(["type", "title", "director", "date_added"])

df_netflix_deduped.show(5)

+-------+-----+--------------------+--------------------+--------------------+--------------------+----------------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|      date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+----------------+------------+------+--------+--------------------+--------------------+
|  s6274|Movie|"Behind ""The Cov...|          Keiko Yagi|                null|Japan, United States| August 25, 2017|        2015| TV-14| 105 min|Documentaries, In...|After a documenta...|
|  s6705|Movie|"Escape from the ...| Wojciech Marczewski|Janusz Gajos, Zbi...|              Poland| October 1, 2019|        1990| TV-MA|  88 min|Comedies, Dramas,...|Artistic rebellio...|
|  s4154|Movie|"Gabriel ""Fluffy...|     Manny Rodriguez|   

In [5]:
(deltaTable_titles.alias('movie_and_show_titles')
 .merge(df_netflix_deduped.alias('updates')
        ,"""lower(movie_and_show_titles.type) = lower(updates.type)
          AND lower(movie_and_show_titles.title) = lower(updates.title)
          AND lower(movie_and_show_titles.director) = lower(updates.director)
          AND movie_and_show_titles.date_added = updates.date_added""")
 .whenMatchedUpdate(set ={
    "show_id": "updates.show_id",
     "type": "updates.type",
     "title" : "updates.title",
     "director" : "updates.director",
     "cast" : "updates.cast",
     "country" : "updates.country",
     "date_added" : "updates.date_added",
     "release_year" : "updates.release_year",
     "rating" : "updates.rating",
     "duration" : "updates.duration",
     "listed_in" : "updates.listed_in",
     "description" : "updates.description"})
 .whenNotMatchedInsert(values = {
    "show_id": "updates.show_id",
     "type": "updates.type",
     "title" : "updates.title",
     "director" : "updates.director",
     "cast" : "updates.cast",
     "country" : "updates.country",
     "date_added" : "updates.date_added",
     "release_year" : "updates.release_year",
     "rating" : "updates.rating",
     "duration" : "updates.duration",
     "listed_in" : "updates.listed_in",
     "description" : "updates.description"})
  .execute())

In [9]:
%%sparksql

DESCRIBE HISTORY "/opt/workspace/data/delta_lake/movie_and_show_titles"

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-05-28 15:19:00.113000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(((lower(type#329) = lower(type#2536)) AND (lower(title#330) = lower(title#2535))) AND (release_year#335 = release_year#2538))""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,1,Serializable,False,"{'numOutputRows': '12484', 'numTargetBytesAdded': '2848903', 'numTargetRowsInserted': '3678', 'numTargetFilesAdded': '2', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '2', 'numTargetRowsMatchedUpdated': '2221', 'executionTimeMs': '1894', 'numTargetRowsCopied': '6585', 'rewriteTimeMs': '897', 'numTargetRowsUpdated': '2221', 'numTargetRowsDeleted': '0', 'scanTimeMs': '771', 'numSourceRows': '5898', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '2027778'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
1,2025-05-28 15:14:42.834000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(((lower(type#329) = lower(type#553)) AND (lower(title#330) = lower(title#554))) AND ((lower(director#331) = lower(director#555)) AND (date_added#334 = date_added#558)))""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,0,Serializable,False,"{'numOutputRows': '8806', 'numTargetBytesAdded': '2027778', 'numTargetRowsInserted': '8806', 'numTargetFilesAdded': '2', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '0', 'numTargetRowsMatchedUpdated': '0', 'executionTimeMs': '2510', 'numTargetRowsCopied': '0', 'rewriteTimeMs': '941', 'numTargetRowsUpdated': '0', 'numTargetRowsDeleted': '0', 'scanTimeMs': '638', 'numSourceRows': '8806', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '0'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
0,2025-05-28 15:14:30.463000,null,null,CREATE OR REPLACE TABLE,"{'description': None, 'partitionBy': '[]', 'properties': '{}', 'isManaged': 'false'}",null,null,null,null,Serializable,True,{},null,Apache-Spark/3.4.1 Delta-Lake/2.4.0


In [7]:
df_titles = (spark.read
             .format("csv")
             .option("header", "true")
             .load("../data/titles.csv"))

df_titles_deduped = df_titles.dropDuplicates(["type", "title"])

df_titles_deduped.show(5)

+--------------------+--------------------+----+-----------+------------+-----------------+-------+------+--------------------+-------+-------+----------+----------+---------------+----------+
|                  id|               title|type|description|release_year|age_certification|runtime|genres|production_countries|seasons|imdb_id|imdb_score|imdb_votes|tmdb_popularity|tmdb_score|
+--------------------+--------------------+----+-----------+------------+-----------------+-------+------+--------------------+-------+-------+----------+----------+---------------+----------+
|It stars the awar...|                null|null|       null|        null|             null|   null|  null|                null|   null|   null|      null|      null|           null|      null|
| Adea‘s uncle is ...| Bibi can't manag...|null|       null|        null|             null|   null|  null|                null|   null|   null|      null|      null|           null|      null|
|Fated to Love You...| Taiwan. It w

In [8]:
df_titles_deduped.createOrReplaceTempView("titles_deduped")

(deltaTable_titles.alias('movie_and_show_titles')
 .merge(df_titles_deduped.alias('updates')
        ,"""lower(movie_and_show_titles.type) = lower(updates.type)
          AND lower(movie_and_show_titles.title) = lower(updates.title)
          AND movie_and_show_titles.release_year = updates.release_year""")
 .whenMatchedUpdate(set ={
     "show_id" : "updates.id",
     "type" : "updates.type",
     "title" : "updates.title",
     "country" : "updates.production_countries",
     "release_year" : "updates.release_year",
     "rating" : "updates.age_certification",
     "duration" : "updates.runtime",
     "listed_in" : "updates.genres",
     "description" : "updates.description"})
 .whenNotMatchedInsert(values = {
     "show_id" : "updates.id",
     "type" : "updates.type",
     "title" : "updates.title",
     "country" : "updates.production_countries",
     "release_year" : "updates.release_year",
     "rating" : "updates.age_certification",
     "duration" : "updates.runtime",
     "listed_in" : "updates.genres",
     "description" : "updates.description"})
  .execute())

In [10]:
spark.stop()